# 04 - Postulate 3: Measurement

**Quantum measurements are described by a collection of measurement operators. The probability of a result is given by the Born rule, and the state collapses to the corresponding eigenstate after measurement.**

In plain English: when you measure a qubit, three things happen:
1. You get a classical result (0 or 1)
2. The probability of each result follows the Born rule: P = |amplitude|^2
3. The qubit state changes (collapses) to match the result

This is the only non-reversible part of quantum computing. Once you measure, the superposition is gone.

In [ ]:
import numpy as np
np.random.seed(42)  # for reproducibility

## The Born Rule

If a qubit is in state `|psi> = alpha|0> + beta|1>`, then:

- Probability of measuring 0: `P(0) = |alpha|^2`
- Probability of measuring 1: `P(1) = |beta|^2`

This is the Born rule. It connects the abstract math (amplitudes) to what we actually observe (probabilities).

In [ ]:
# A qubit in superposition
alpha = np.sqrt(0.7)
beta = np.sqrt(0.3) * np.exp(1j * np.pi/4)  # with some phase

psi = np.array([alpha, beta], dtype=complex)
psi = psi / np.linalg.norm(psi)  # normalize just to be safe

# Born rule
p0 = abs(psi[0])**2
p1 = abs(psi[1])**2

print(f"State |psi> = {psi.round(4)}")
print(f"P(0) = |alpha|^2 = {p0:.4f}")
print(f"P(1) = |beta|^2  = {p1:.4f}")
print(f"Total = {p0 + p1:.4f}")

## Simulating Measurement

A single measurement gives ONE random result. To see the probability distribution, you need to repeat the experiment many times. This is exactly what happens on real quantum hardware.

In [ ]:
def measure(state, num_shots=1):
    """Simulate measuring a qubit state."""
    probabilities = np.abs(state)**2
    results = np.random.choice(len(state), size=num_shots, p=probabilities)
    return results

# Single measurement -- random
result = measure(psi, 1)
print(f"Single measurement: {result[0]}")
print("Run this cell multiple times -- you will get different results.\n")

# Many measurements -- statistics converge to Born rule
shots = 10000
results = measure(psi, shots)
count_0 = np.sum(results == 0)
count_1 = np.sum(results == 1)

print(f"{shots} measurements:")
print(f"  Got 0: {count_0} times ({count_0/shots:.4f})")
print(f"  Got 1: {count_1} times ({count_1/shots:.4f})")
print(f"\nExpected (Born rule): P(0)={p0:.4f}, P(1)={p1:.4f}")
print("The more shots you take, the closer the results get to the Born rule.")

## Wave Function Collapse

After measurement, the state collapses to the basis state corresponding to the result.

- If you measure 0, the state becomes |0>
- If you measure 1, the state becomes |1>

The superposition is destroyed. If you measure again, you get the same result with 100% certainty.

In [ ]:
# Before measurement
psi = np.array([1/np.sqrt(2), 1/np.sqrt(2)], dtype=complex)  # |+>
print(f"Before measurement: {psi}")
print(f"  P(0) = {abs(psi[0])**2:.2f}, P(1) = {abs(psi[1])**2:.2f}")

# Simulate a measurement
result = measure(psi, 1)[0]
print(f"\nMeasurement result: {result}")

# After measurement: collapsed state
if result == 0:
    psi_collapsed = np.array([1, 0], dtype=complex)
else:
    psi_collapsed = np.array([0, 1], dtype=complex)

print(f"After collapse: {psi_collapsed}")
print(f"  P(0) = {abs(psi_collapsed[0])**2:.2f}, P(1) = {abs(psi_collapsed[1])**2:.2f}")
print("\nThe superposition is gone. Measuring again gives the same result every time.")

## Measurement in Different Bases

The computational basis {|0>, |1>} is not the only basis you can measure in. You can measure in any orthonormal basis.

To measure in a different basis, apply a change-of-basis unitary before measuring in the computational basis.

- Measure in X basis: apply H, then measure
- Measure in Y basis: apply S^dagger then H, then measure

In [ ]:
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

# State |0> measured in computational (Z) basis: always 0
ket_0 = np.array([1, 0], dtype=complex)
print("State |0> measured in Z basis:")
print(f"  P(0) = {abs(ket_0[0])**2:.2f}, P(1) = {abs(ket_0[1])**2:.2f}")

# Same state measured in X basis: apply H first
ket_0_in_x = H @ ket_0  # change to X basis
print("\nState |0> measured in X basis (apply H then measure):")
print(f"  P(+) = {abs(ket_0_in_x[0])**2:.2f}, P(-) = {abs(ket_0_in_x[1])**2:.2f}")

# State |+> measured in Z basis: 50/50
ket_plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
print("\nState |+> measured in Z basis:")
print(f"  P(0) = {abs(ket_plus[0])**2:.2f}, P(1) = {abs(ket_plus[1])**2:.2f}")

# State |+> measured in X basis: always +
ket_plus_in_x = H @ ket_plus
print("\nState |+> measured in X basis:")
print(f"  P(+) = {abs(ket_plus_in_x[0])**2:.2f}, P(-) = {abs(ket_plus_in_x[1])**2:.2f}")

## Measurement with Projection Operators

Mathematically, measuring in the computational basis uses projection operators:

- P_0 = |0><0| (projects onto |0>)
- P_1 = |1><1| (projects onto |1>)

Probability of outcome k: `P(k) = <psi|P_k|psi>`

State after getting outcome k: `P_k|psi> / sqrt(P(k))`

In [ ]:
# Projection operators
ket_0 = np.array([[1], [0]], dtype=complex)
ket_1 = np.array([[0], [1]], dtype=complex)

P0 = ket_0 @ ket_0.conj().T  # |0><0|
P1 = ket_1 @ ket_1.conj().T  # |1><1|

# State to measure
psi = np.array([[1/np.sqrt(3)], [np.sqrt(2/3)]], dtype=complex)

# Probability of measuring 0
prob_0 = (psi.conj().T @ P0 @ psi)[0, 0].real
print(f"P(0) = <psi|P0|psi> = {prob_0:.4f}")

# Probability of measuring 1 
prob_1 = (psi.conj().T @ P1 @ psi)[0, 0].real
print(f"P(1) = <psi|P1|psi> = {prob_1:.4f}")
print(f"Total = {prob_0 + prob_1:.4f}")

# Post-measurement state if we got 0
post_state_0 = (P0 @ psi) / np.sqrt(prob_0)
print(f"\nState after measuring 0: {post_state_0.flatten().round(4)}")
print("This is |0>, as expected.")

## The No-Cloning Theorem

You cannot copy an unknown quantum state. This is a direct consequence of the linearity of quantum mechanics.

If copying were possible, you could:
1. Make many copies of a state
2. Measure each copy in different bases
3. Learn everything about the state

But quantum mechanics says you cannot know everything (uncertainty principle). So copying must be impossible.

This is why quantum cryptography works -- an eavesdropper cannot copy quantum messages.

In [ ]:
# Can we build a "cloning gate" that does |psi>|0> --> |psi>|psi> ?
# Let us try and show it must fail.

# If cloning works for |0>: |0>|0> --> |0>|0>  (trivially works)
# If cloning works for |1>: |1>|0> --> |1>|1>  (also works)
# But for |+> = (|0>+|1>)/sqrt(2):
#   By linearity: |+>|0> --> (|0>|0> + |1>|1>) / sqrt(2) = Bell state
#   But we WANTED: |+>|+> = (|0>+|1>)(|0>+|1>)/2 = (|00>+|01>+|10>+|11>)/2
#   
#   Bell state != |+>|+>
#   Contradiction. Cloning does not work.

ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

# What linearity gives us
bell = (np.kron(ket_0, ket_0) + np.kron(ket_1, ket_1)) / np.sqrt(2)

# What we wanted (|+>|+>)
ket_plus = (ket_0 + ket_1) / np.sqrt(2)
plus_plus = np.kron(ket_plus, ket_plus)

print(f"What linearity gives: {bell.round(4)}")
print(f"What we wanted:      {plus_plus.round(4)}")
print(f"\nAre they equal? {np.allclose(bell, plus_plus)}")
print("No. Cloning is impossible for unknown quantum states.")

## Key Takeaway

- Born rule: probability = |amplitude|^2
- Measurement collapses the state -- superposition is destroyed
- Measurement is irreversible (unlike gates)
- You can measure in any basis by applying a unitary first
- You cannot copy quantum states (no-cloning theorem)

## Exercises

1. Create the state `|psi> = (sqrt(3)/2)|0> + (1/2)|1>`. Compute P(0) and P(1) using the Born rule. Simulate 10000 measurements and compare.

2. Measure the state |-> in the Z basis, then in the X basis. In which basis is the outcome certain?

3. Create a state and measure it. After collapse, apply a Hadamard and measure again. Is the result of the second measurement determined by the first?

In [ ]:
# Your code here
